### Install Libraries

### Import Libraries

In [1]:
# from langchain_community.llms import CTransformers
# from langchain_community.llms import LlamaCpp # <- llamaCpp! An Alternate option for CTransformers - Make a Poll.
# from langchain.callbacks.manager import CallbackManager
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate

from langchain.chains import ConversationChain
from langchain.chains.conversation.memory import (ConversationBufferMemory, 
                                                  ConversationSummaryMemory, 
                                                  ConversationBufferWindowMemory,
                                                  ConversationKGMemory)
# RAG 1st
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.document_loaders import TextLoader
from langchain_community.embeddings.sentence_transformer import (
    SentenceTransformerEmbeddings,
)

from langchain.storage import InMemoryStore
from langchain_community.document_loaders import TextLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.retrievers import ParentDocumentRetriever
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

### Retriever

In [2]:
loader = PyMuPDFLoader(".\\Data\\PDFs\\DepressionGuide-web.pdf")
documents  = loader.load()

In [3]:
embedding_function = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

c:\Users\User\anaconda3\envs\omdena_chatbot\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=2000)

child_splitter = RecursiveCharacterTextSplitter(chunk_size=400)

vectorstore = Chroma(collection_name="split_parents", embedding_function=embedding_function)

store = InMemoryStore()

c:\Users\User\anaconda3\envs\omdena_chatbot\Lib\site-packages\onnxruntime\capi\onnxruntime_validation.py:26: UserWarning: Unsupported Windows version (11). ONNX Runtime supports Windows 10 and above, only.
  warnings.warn(


In [5]:
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

In [6]:
retriever.add_documents(documents)

In [7]:
# Testing
retriever.get_relevant_documents("I'm Tired all the time, feeling “lazy”")

c:\Users\User\anaconda3\envs\omdena_chatbot\Lib\site-packages\langchain_core\_api\deprecation.py:119: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 0.3.0. Use invoke instead.
  warn_deprecated(


[Document(page_content='Depression: Parents’ Medication Guide       5\nCauses and Symptoms\nWhy does my child \nhave depression?\nWe don’t fully understand all the \ncauses of depression; we think it’s a \ncombination of genetics (inherited traits) \nand environmental factors (events and \nsurroundings). There is no single cause. \nStressors or events that cause a stressful \nresponse and genetic factors can cause \ndepression. Stressors can be triggers \nthat result from pediatric illnesses and \ndiseases, such as viral infections; diseases \nof the thyroid and endocrine system; head \ninjury; epilepsy; and heart, kidney, and lung \ndiseases. A family history of depression \nis a major genetic factor; a child can be \nmore prone to becoming depressed if \na parent or sibling has been diagnosed \nwith depression. Stressors in everyday \nlife also contribute to the development \nof depression, for example, the loss of a \nclose loved one; parents frequently arguing, \nseparating, or div

In [36]:
question = "I've been feeling really anxious lately, especially with everything going on at work. Do you have any advice on how I can manage my stress better?"

retrieved_context = retriever.get_relevant_documents(question)

In [37]:
retrieved_context

[Document(page_content='Depression: Parents’ Medication Guide       11\nsyndrome occurs when high levels of \nserotonin accumulate in the body, and \nit most often happens when a person is \ntaking more than one medication that \naffects the serotonin level. Symptoms of \nserotonin syndrome may include fever, \nconfusion, tremor, restlessness, sweating, \nand increased reflexes.\nOther medications, in addition to those \nthat affect serotonin, can interact with \nSSRIs and other antidepressants and \ncause problems. Therefore, it is very \nimportant that you tell your child’s \ndoctor about all the medications and \nsupplements that your child takes. It is \nalso important to discuss with your child’s \ndoctor any new supplements or over-\nthe-counter medications or medications \nprescribed to your child by other doctors \nbefore taking those medications.\nHow can I help monitor my \nchild during treatment?\nBecause some youth have adverse \nphysical and/or emotional reactions to \nant

In [39]:
cleaned_retrieved_context = []
for i in range(len(retrieved_context)):
    cleaned_retrieved_context.append(retrieved_context[i].page_content)

In [40]:
cleaned_retrieved_context

['Depression: Parents’ Medication Guide       11\nsyndrome occurs when high levels of \nserotonin accumulate in the body, and \nit most often happens when a person is \ntaking more than one medication that \naffects the serotonin level. Symptoms of \nserotonin syndrome may include fever, \nconfusion, tremor, restlessness, sweating, \nand increased reflexes.\nOther medications, in addition to those \nthat affect serotonin, can interact with \nSSRIs and other antidepressants and \ncause problems. Therefore, it is very \nimportant that you tell your child’s \ndoctor about all the medications and \nsupplements that your child takes. It is \nalso important to discuss with your child’s \ndoctor any new supplements or over-\nthe-counter medications or medications \nprescribed to your child by other doctors \nbefore taking those medications.\nHow can I help monitor my \nchild during treatment?\nBecause some youth have adverse \nphysical and/or emotional reactions to \nantidepressants, parents 

### Augmentor

In [33]:
instructions = "You are an empathetic and knowledgeable mental health expert chatbot. You have access to the following context and chat history to provide thoughtful and supportive responses to the patient."

In [34]:
memory = ConversationBufferWindowMemory(input_key="question", memory_key="history", return_messages=True, k=5)

In [35]:
history = memory.load_memory_variables({})['history']

In [41]:
prompt = instructions + "\n\n" + "Context: " + str(cleaned_retrieved_context) + "\n\n" + "Chat History: " +str(history) + "\n\n" + "Question: " +  question + "\n\n" + "Answer:"

In [42]:
print(prompt)

You are an empathetic and knowledgeable mental health expert chatbot. You have access to the following context and chat history to provide thoughtful and supportive responses to the patient.

Context: ['Depression: Parents’ Medication Guide       11\nsyndrome occurs when high levels of \nserotonin accumulate in the body, and \nit most often happens when a person is \ntaking more than one medication that \naffects the serotonin level. Symptoms of \nserotonin syndrome may include fever, \nconfusion, tremor, restlessness, sweating, \nand increased reflexes.\nOther medications, in addition to those \nthat affect serotonin, can interact with \nSSRIs and other antidepressants and \ncause problems. Therefore, it is very \nimportant that you tell your child’s \ndoctor about all the medications and \nsupplements that your child takes. It is \nalso important to discuss with your child’s \ndoctor any new supplements or over-\nthe-counter medications or medications \nprescribed to your child by ot

In [43]:
import os

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = "REDACTED_GOOGLE_API_KEY"

In [44]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash-latest")

In [45]:
result = llm.invoke(prompt)

In [46]:
print(result.content)

It's understandable that you're feeling anxious, especially with the added stress of work. It sounds like you're looking for some coping mechanisms to help you manage your stress. 

It's great that you're seeking support! Here are some things you might try:

* **Mindfulness and relaxation techniques:** Simple things like deep breathing exercises, meditation, or even just taking a few minutes to focus on your senses can help calm your mind and body. 
* **Regular exercise:** Physical activity can be a great stress reliever. Aim for at least 30 minutes of moderate-intensity exercise most days of the week.
* **Healthy diet and sleep:**  Make sure you're eating nutritious foods and getting enough sleep.  Both of these contribute to your overall well-being and can help you better manage stress.
* **Identify and address stressors:** Think about what specific situations or tasks are causing you the most anxiety. Can you break them down into smaller, more manageable steps? Can you delegate some

In [1]:
memory.chat_memory.add_user_message(question)

NameError: name 'memory' is not defined

In [8]:
# from langchain import PromptTemplate
# from langchain.prompts.chat import (
#     ChatPromptTemplate,
#     SystemMessagePromptTemplate,
#     AIMessagePromptTemplate,
#     HumanMessagePromptTemplate,
# )

# # Define system and user message templates
# system_message_template = '''You are a Mental Health Specialist (therapist).
# Your job is to provide support for individuals with Depressive Disorder.
# Act as a compassionate listener and offer helpful responses based on the user's queries.
# If the user seeks casual conversation, be friendly and supportive.
# If they seek factual information, use the context of the conversation to provide relevant responses.
# If unsure, be honest and say, 'This is out of the scope of my knowledge.' Always respond directly to the user's query without deviation.
# Context: {context} '''

# system_message_template = "You are a professional therapist, act like one., Here's the Question : {question}, Previous Context: {context}"

# user_message_template = "User Query: {question} Answer:"

# # Create message templates
# system_message = SystemMessagePromptTemplate.from_template(system_message_template)
# user_message = HumanMessagePromptTemplate.from_template(user_message_template)

# # Compile messages into a chat prompt template
# messages = [system_message, user_message]
# chatbot_prompt = ChatPromptTemplate.from_messages(messages)

In [9]:
# chatbot_prompt

In [10]:
# # aka custom_template

# condense_question_prompt = """Given the following conversation and a follow-up message, \
# rephrase the follow-up message to a stand-alone question or instruction that \
# represents the user's intent, add all context needed if necessary to generate a complete and \
# unambiguous question or instruction, only based on the history, don't make up messages. \
# Maintain the same language as the follow up input message.

# Chat History:
# {chat_history}

# Follow Up Input: {question}
# Standalone question or instruction:"""

### Generator

In [2]:
import os

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = "REDACTED_GOOGLE_API_KEY"

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash-latest")

# result = llm.invoke("Write a ballad about LangChain")
# print(result.content)

c:\Users\User\Documents\aiml\Mentalbot\Mentalbot\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
from langchain.prompts import PromptTemplate

prompt = PromptTemplate(
    input_variables=["product"],
    template="What is a good name for a company that makes {product}?",
)

In [7]:
from langchain.chains import LLMChain
chain = LLMChain(llm=llm, prompt=prompt, verbose=True)

print(chain.run("gaming laptop"))

C:\Users\User\AppData\Local\Temp\ipykernel_12540\3833003751.py:2: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use RunnableSequence, e.g., `prompt | llm` instead.
  chain = LLMChain(llm=llm, prompt=prompt, verbose=True)
C:\Users\User\AppData\Local\Temp\ipykernel_12540\3833003751.py:4: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use invoke instead.
  print(chain.run("gaming laptop"))




> Entering new LLMChain chain...
Prompt after formatting:
What is a good name for a company that makes gaming laptop?

> Finished chain.
Here are some good names for a company that makes gaming laptops, categorized by style:

**Bold & Aggressive:**

* **Apex Predator**
* **FuryTech**
* **Warfare Labs**
* **Ironclad Gaming**
* **Crimson Forge**

**Sophisticated & Modern:**

* **Zenith Gaming**
* **Apex Systems**
* **Lucid Tech**
* **Quantum Core**
* **Elevate Gaming**

**Unique & Playful:**

* **Pixel Forge**
* **The Glitch**
* **Byte Brigade**
* **Level Up Labs**
* **GameChanger**

**Descriptive & Straightforward:**

* **Elite Gaming Laptops**
* **Pro Series Gaming**
* **Apex Performance Laptops**
* **Gamer's Edge**
* **Ultimate Gaming Systems**

**Tips for Choosing a Name:**

* **Keep it short and memorable.**
* **Make sure it's easy to pronounce and spell.**
* **Consider your target audience.**
* **Check for trademark availability.**
* **Try saying the name out loud to see how it s

In [12]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")

In [13]:
vector = embeddings.embed_query("hello, world!")
vector[:5]

[0.05168594419956207,
 -0.030764883384108543,
 -0.03062233328819275,
 -0.02802734449505806,
 0.01813092641532421]

### Orchestrator

In [ ]:
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ChatMessageHistory,ConversationSummaryBufferMemory,ConversationBufferMemory

In [ ]:
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

In [ ]:
# memory = ConversationSummaryBufferMemory(
#         memory_key="chat_history",
#         input_key="question",
#         llm=llm,
#         max_token_limit=40,
#         return_messages=True
#     )

In [ ]:
import torch

In [ ]:
# chain = ConversationalRetrievalChain.from_llm(llm = llm,
#                                               retriever=retriever,
#                                               memory = memory,
#                                               rephrase_question=False)

In [ ]:
# torch.cuda.is_available()

In [ ]:
# qa = ConversationalRetrievalChain.from_llm(
#     llm,
#     retriever=retriever,
#     memory = memory,
#     return_source_documents=False,
#     chain_type="stuff",
#     max_tokens_limit=100, # Llama-2 max = 4096
#     # condense_question_prompt= PromptTemplate.from_template(condense_question_prompt),
#     combine_docs_chain_kwargs={'prompt': chatbot_prompt},
#     verbose=True,
#     return_generated_question=False,
# )

### ChatBot

In [ ]:
# chain.memory.buffer

In [ ]:
history = ChatMessageHistory()

In [ ]:
history

In [ ]:
# def ask(question: str):
#     answer = qa({"question": question,"chat_history":history.messages})["answer"]
#     print("##------##")
#     # print(answer)
#     return answer

# ask("I'm Tired all the time, feeling “lazy”")
# ask("I Think Its because of Social Media as I am a Socia Media Influencer.")

In [ ]:
question = "I'm Tired all the time, feeling lazy"
result = chain({"question": question, "chat_history": history.messages})

In [ ]:
from langchain.memory import ConversationBufferWindowMemory

In [ ]:
memory = ConversationBufferWindowMemory( k=2)
memory.save_context({"input": "hi"}, {"output": "whats up"})
memory.save_context({"input": "not much you"}, {"output": "not much"})

In [ ]:
memory.load_memory_variables({})

In [ ]:
memory = ConversationBufferWindowMemory(input_key="question", memory_key="history", return_messages=True, k=5)

In [ ]:
memory.save_context({"question": "hi"}, {"output": "whats up"})
memory.save_context({"question": "not much you"}, {"output": "not much"})

In [ ]:
history = memory.load_memory_variables({})['history']

In [ ]:
history[0].content

In [ ]:
memory.chat_memory.add_user_message

In [ ]:
from langchain.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=vectordb.as_retriever()
)